# DocTrust-VLM — 50-document, two-model Colab comparison

This notebook compares **HuggingFaceTB/SmolVLM-Instruct (2.2B)** with **Qwen/Qwen2.5-VL-3B-Instruct** on the same 50 DocVQA-derived documents and five paired image conditions.

The complete run is **500 predictions**: 50 documents × 5 variants × 2 models. It stores raw predictions, latency, peak VRAM, exact model revisions, manifests, configs, environment metadata, metrics and a comparison table.

**Before starting:** Runtime → Change runtime type → choose a GPU. A T4-class 16 GB GPU is the target. Run one model at a time; the notebook unloads SmolVLM before loading Qwen.


## 1. Clone or update the public repository

No token or Colab secret is required. Rerun this cell whenever the repository has been updated.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/TharunChougoni/doctrust-vlm.git"
REPO_DIR = Path("/content/doctrust-vlm")
if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    shutil.rmtree(REPO_DIR)
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())


## 2. Install Colab dependencies

Colab already supplies CUDA-enabled PyTorch. A warning about preinstalled Gradio and `huggingface-hub` can be ignored because this experiment does not use Gradio.


In [ ]:
%pip install -q -r requirements-colab.txt


## 3. Verify GPU

Both models run in FP16, batch size 1. Stop if total VRAM is below 10 GB.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Select Runtime → Change runtime type → GPU.")
props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / 1024**3
print("GPU:", props.name)
print(f"Total VRAM: {total_gb:.1f} GB")
if total_gb < 10:
    raise RuntimeError("This two-model FP16 workflow requires at least 10 GB total VRAM.")


## 4. Fetch a deterministic 50-document subset

The source mirror supplies images, accepted answers, OCR words, character offsets and OCR boxes. The fetcher keeps unique documents, requires OCR answer-match confidence ≥ 0.95 and derives a proposed answer-evidence box.

Raw dataset images stay untracked. **Every proposed box still requires visual audit.**


In [ ]:
import subprocess
import sys

N_EXAMPLES = 50
SCAN_LIMIT = 300
subprocess.run(
    [
        sys.executable,
        "scripts/fetch_docvqa_samples.py",
        "--count", str(N_EXAMPLES),
        "--scan", str(SCAN_LIMIT),
        "--acknowledge-docvqa-terms",
    ],
    check=True,
)


## 5. Visual audit of all 50 evidence boxes

The notebook displays ten documents per figure to keep Colab responsive. Each red rectangle must cover the printed accepted answer. If a box is wrong, correct or exclude that row before continuing.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

source_rows = [
    json.loads(line)
    for line in Path("data/manifests/source.jsonl").read_text().splitlines()
]
assert len(source_rows) == N_EXAMPLES, (len(source_rows), N_EXAMPLES)

PAGE_SIZE = 10
for page_start in range(0, len(source_rows), PAGE_SIZE):
    page = source_rows[page_start:page_start + PAGE_SIZE]
    fig, axes = plt.subplots(5, 2, figsize=(16, 38))
    axes = list(axes.flat)
    for ax, row in zip(axes, page):
        image = Image.open(row["image_path"]).convert("RGB")
        x1, y1, x2, y2 = row["evidence_box"]
        draw = ImageDraw.Draw(image)
        draw.rectangle(
            (x1 * image.width, y1 * image.height, x2 * image.width, y2 * image.height),
            outline="red",
            width=max(4, image.width // 250),
        )
        ax.imshow(image)
        ax.set_title(
            f"{row['id']} | confidence={row.get('evidence_match_confidence', 'n/a')}"
            f"\nQ: {row['question']}\nA: {row['answers'][0]}",
            fontsize=9,
        )
        ax.axis("off")
    for ax in axes[len(page):]:
        ax.axis("off")
    fig.suptitle(
        f"Evidence audit {page_start + 1}–{page_start + len(page)} of {len(source_rows)}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()


### Audit gate

Change `BOXES_AUDITED` to `True` only after checking all five figures. This is the main quality-control step when scaling beyond ten examples.


In [ ]:
BOXES_AUDITED = False
assert BOXES_AUDITED, "Audit all 50 evidence boxes, then set BOXES_AUDITED=True."


## 6. Generate and verify all 250 paired images

For each document: clean, JPEG, blur, distractor occlusion and evidence occlusion. Preparation clears results from an older dataset/config. Once inference starts, rerun inference cells—not this cell—to resume without losing progress.


In [ ]:
import sys
from collections import defaultdict

from PIL import Image, ImageChops

sys.path.insert(0, str(Path("src").resolve()))
from doctrust.prepare import prepare

prepared_path = prepare("configs/colab_smolvlm_2b.yaml")
prepared_rows = [json.loads(line) for line in prepared_path.read_text().splitlines()]
assert len(prepared_rows) == N_EXAMPLES * 5

# A new preparation invalidates every previous prediction/comparison artifact.
for stale_path in [
    Path("results/colab_predictions.jsonl"),
    Path("results/colab_metrics.json"),
    Path("results/colab_error_analysis.jsonl"),
    Path("results/qwen_predictions.jsonl"),
    Path("results/qwen_metrics.json"),
    Path("results/qwen_error_analysis.jsonl"),
    Path("results/model_comparison.csv"),
    Path("results/model_comparison.json"),
]:
    stale_path.unlink(missing_ok=True)

# Fail if any declared corruption silently becomes a clean-image no-op.
by_source = defaultdict(dict)
for row in prepared_rows:
    by_source[row["source_id"]][row["variant"]] = row
for source_id, variants in by_source.items():
    clean = Image.open(variants["clean"]["image_path"]).convert("RGB")
    for variant_name, row in variants.items():
        if variant_name == "clean":
            continue
        transformed = Image.open(row["image_path"]).convert("RGB")
        if ImageChops.difference(clean, transformed).getbbox() is None:
            raise RuntimeError(f"{source_id}/{variant_name} is identical to clean")

print("Prepared:", len(prepared_rows), "images")
print("Verified: all 200 non-clean images differ from their clean image")
print("Cleared stale prediction artifacts")

# Preview the five actual model inputs for the first source.
first_source = prepared_rows[0]["source_id"]
preview_rows = [row for row in prepared_rows if row["source_id"] == first_source]
fig, axes = plt.subplots(1, 5, figsize=(20, 6))
for ax, row in zip(axes, preview_rows):
    ax.imshow(Image.open(row["image_path"]))
    ax.set_title(row["variant"])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 7. Record reproducibility metadata

This captures environment versions, GPU, Git commit, seed and dataset size. Model revisions are added after each model loads and are also written into every prediction row.


In [ ]:
import datetime as dt
import importlib.metadata as metadata
import platform

Path("results").mkdir(exist_ok=True)
def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

run_metadata = {
    "started_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "python": platform.python_version(),
    "gpu": props.name,
    "gpu_total_vram_gb": total_gb,
    "cuda": torch.version.cuda,
    "packages": {
        name: package_version(name)
        for name in ["torch", "transformers", "accelerate", "huggingface-hub", "qwen-vl-utils"]
    },
    "seed": 17,
    "source_count": len(source_rows),
    "variants_per_source": 5,
    "models": {},
}
Path("results/run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
Path("results/pip_freeze.txt").write_text(freeze)
print(json.dumps(run_metadata, indent=2))


## 8. SmolVLM 2.2B: load, infer and evaluate

Predictions are appended after every image. If interrupted, rerun only the inference cell to skip completed IDs.


In [ ]:
from doctrust.config import load_config
from doctrust.modeling import DocumentVLM

smol_config_path = "configs/colab_smolvlm_2b.yaml"
smol_config = load_config(smol_config_path)
print("Loading:", smol_config["model"]["model_id"])
smol_model = DocumentVLM(smol_config["model"])
run_metadata["models"][smol_model.model_id] = {
    "revision": smol_model.model_revision,
    "config": smol_config_path,
}
Path("results/run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")
print("Loaded on:", smol_model.model.device, "revision:", smol_model.model_revision)


In [ ]:
from doctrust.io import read_jsonl
from doctrust.run import run

smol_predictions_path = run(smol_config_path, model=smol_model)
smol_predictions = read_jsonl(smol_predictions_path)
print("SmolVLM predictions:", len(smol_predictions), "of", N_EXAMPLES * 5)


In [ ]:
from doctrust.evaluate import evaluate

smol_metrics = evaluate(smol_predictions_path)
smol_metrics_path = Path(smol_config["output"]["metrics"])
smol_metrics_path.write_text(json.dumps(smol_metrics, indent=2) + "\n")
print(json.dumps(smol_metrics, indent=2))


### Inspect SmolVLM outputs

The table is the raw experiment record. Check evidence-occlusion hallucinations, not only aggregate means.


In [ ]:
import pandas as pd

smol_table = pd.DataFrame(smol_predictions)[
    ["source_id", "variant", "question", "answers", "prediction", "latency_seconds", "peak_vram_mb"]
]
display(smol_table)


## 9. Unload SmolVLM before Qwen

Do not keep both models in GPU memory simultaneously.


In [ ]:
import gc

del smol_model
gc.collect()
torch.cuda.empty_cache()
print(f"Allocated after unload: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 10. Qwen2.5-VL 3B: load, infer and evaluate

Qwen2.5-VL-3B is a strong document model; its model card reports 93.9 on DocVQA test. We use the identical images, questions, prompt policy, seed and short-answer limit as SmolVLM.


In [ ]:
qwen_config_path = "configs/colab_qwen2_5_vl_3b.yaml"
qwen_config = load_config(qwen_config_path)
print("Loading:", qwen_config["model"]["model_id"])
qwen_model = DocumentVLM(qwen_config["model"])
run_metadata["models"][qwen_model.model_id] = {
    "revision": qwen_model.model_revision,
    "config": qwen_config_path,
}
Path("results/run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")
print("Loaded on:", qwen_model.model.device, "revision:", qwen_model.model_revision)


In [ ]:
qwen_predictions_path = run(qwen_config_path, model=qwen_model)
qwen_predictions = read_jsonl(qwen_predictions_path)
print("Qwen predictions:", len(qwen_predictions), "of", N_EXAMPLES * 5)


In [ ]:
qwen_metrics = evaluate(qwen_predictions_path)
qwen_metrics_path = Path(qwen_config["output"]["metrics"])
qwen_metrics_path.write_text(json.dumps(qwen_metrics, indent=2) + "\n")
print(json.dumps(qwen_metrics, indent=2))


## 11. Side-by-side model statistics

Clean ANLS measures task ability. Nuisance robustness is reported conditionally on each model's clean-correct subset. Evidence occlusion is judged by abstention/false-answer rate, not ANLS.


In [ ]:
import numpy as np
from doctrust.metrics import anls, is_abstention

def comparison_row(label, metrics, predictions):
    variants = metrics["variants"]
    latencies = [float(row["latency_seconds"]) for row in predictions]
    peaks = [float(row["peak_vram_mb"]) for row in predictions]
    return {
        "model": label,
        "predictions": len(predictions),
        "clean_correct": metrics["clean_correct_source_count"],
        "clean_mean_anls": variants["clean"]["mean_anls"],
        "jpeg_conditional_anls": variants["jpeg_q35"]["conditional_mean_anls"],
        "blur_conditional_anls": variants["blur_1_5"]["conditional_mean_anls"],
        "distractor_conditional_anls": variants["distractor_occlusion"]["conditional_mean_anls"],
        "evidence_abstention_rate": variants["evidence_occlusion"]["abstention_rate"],
        "evidence_conditional_abstention_rate": variants["evidence_occlusion"]["conditional_abstention_rate"],
        "evidence_conditional_false_answer_rate": variants["evidence_occlusion"]["conditional_false_answer_rate"],
        "mean_latency_seconds": sum(latencies) / len(latencies),
        "peak_vram_mb": max(peaks),
    }

comparison = pd.DataFrame([
    comparison_row("SmolVLM-2.2B", smol_metrics, smol_predictions),
    comparison_row("Qwen2.5-VL-3B", qwen_metrics, qwen_predictions),
])
comparison.to_csv("results/model_comparison.csv", index=False)
Path("results/model_comparison.json").write_text(
    json.dumps(comparison.to_dict(orient="records"), indent=2) + "\n"
)
display(comparison)

# Paired bootstrap on the same 50 source documents.
def variant_by_source(predictions, variant):
    return {
        row["source_id"]: row
        for row in predictions
        if row["variant"] == variant
    }

def bootstrap_mean_ci(values, seed=17, replicates=10_000):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(values), size=(replicates, len(values)))
    sampled_means = values[indices].mean(axis=1)
    return {
        "mean_difference": float(values.mean()),
        "ci95_low": float(np.quantile(sampled_means, 0.025)),
        "ci95_high": float(np.quantile(sampled_means, 0.975)),
        "paired_source_count": int(len(values)),
        "bootstrap_replicates": replicates,
        "seed": seed,
    }

smol_clean = variant_by_source(smol_predictions, "clean")
qwen_clean = variant_by_source(qwen_predictions, "clean")
common_ids = sorted(set(smol_clean) & set(qwen_clean))
assert len(common_ids) == N_EXAMPLES
clean_differences = [
    anls(qwen_clean[source_id]["prediction"], qwen_clean[source_id]["answers"])
    - anls(smol_clean[source_id]["prediction"], smol_clean[source_id]["answers"])
    for source_id in common_ids
]

smol_evidence = variant_by_source(smol_predictions, "evidence_occlusion")
qwen_evidence = variant_by_source(qwen_predictions, "evidence_occlusion")
abstention_differences = [
    float(is_abstention(qwen_evidence[source_id]["prediction"]))
    - float(is_abstention(smol_evidence[source_id]["prediction"]))
    for source_id in common_ids
]
paired_bootstrap = {
    "difference_direction": "Qwen2.5-VL-3B minus SmolVLM-2.2B",
    "clean_anls": bootstrap_mean_ci(clean_differences),
    "evidence_abstention_rate": bootstrap_mean_ci(abstention_differences),
    "note": "Intervals quantify uncertainty for this audited 50-document subset, not DocVQA-wide generalization.",
}
Path("results/paired_bootstrap.json").write_text(
    json.dumps(paired_bootstrap, indent=2) + "\n"
)
print(json.dumps(paired_bootstrap, indent=2))


### Inspect Qwen evidence-removal behavior

Display all Qwen evidence-occlusion rows and compare false answers with the model's clean answer.


In [ ]:
qwen_table = pd.DataFrame(qwen_predictions)
display(
    qwen_table[qwen_table["variant"] == "evidence_occlusion"]
    [["source_id", "question", "answers", "prediction", "latency_seconds", "peak_vram_mb"]]
)


## 12. Download complete reproducibility artifacts

The ZIP includes raw predictions for both models, model revisions, package versions, exact configs, source/prepared manifests, metrics and comparison tables. Dataset images are intentionally omitted.


In [ ]:
from google.colab import files

run_metadata["finished_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
Path("results/run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")

artifact_dir = Path("/content/doctrust-artifacts")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
artifact_dir.mkdir()
artifact_paths = [
    Path("data/manifests/source.jsonl"),
    Path("data/manifests/source_provenance.json"),
    Path("data/manifests/prepared_colab.jsonl"),
    Path("configs/colab_smolvlm_2b.yaml"),
    Path("configs/colab_qwen2_5_vl_3b.yaml"),
    Path("results/colab_predictions.jsonl"),
    Path("results/colab_metrics.json"),
    Path("results/qwen_predictions.jsonl"),
    Path("results/qwen_metrics.json"),
    Path("results/model_comparison.csv"),
    Path("results/model_comparison.json"),
    Path("results/paired_bootstrap.json"),
    Path("results/run_metadata.json"),
    Path("results/pip_freeze.txt"),
]
for path in artifact_paths:
    if not path.exists():
        raise FileNotFoundError(f"Missing required artifact: {path}")
    destination = artifact_dir / str(path).replace("/", "__")
    shutil.copy2(path, destination)
archive = shutil.make_archive("/content/doctrust-two-model-artifacts", "zip", artifact_dir)
print("Artifact archive:", archive)
files.download(archive)


## Interpretation checklist

- Report clean ANLS and clean-correct count before robustness.
- Compare nuisance variants only on each model's clean-correct subset.
- A false answer after evidence removal is an evidence-grounding failure.
- Fifty audited documents and 500 predictions are credible for a course/deadline MVP, not a publication benchmark.
- OCR-derived boxes are manually audited approximations, not official ground-truth evidence annotations.
- Preserve the downloaded ZIP; it is the complete experiment record.
